In [1]:
import sys
import numpy as np
import tensorflow as tf
import tensorflow.keras as K
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

2026-02-20 06:03:55.659481: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-20 06:03:55.668041: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-20 06:03:55.677346: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-20 06:03:55.680198: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-20 06:03:55.687745: I tensorflow/core/platform/cpu_feature_guar

In [ ]:
ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

In [3]:
hidden_layers=[1024, 1024]
epochs=1000
act_func=tf.nn.relu
input_dropout=0.2
hidden_dropout=0.5
learning_rate=0.0001
norm='norm'

In [ ]:
train_features, val_features, _, test_features, train_targets, val_targets, _, test_targets = load(norm=norm)

print("Training features shape:", train_features.shape)
print("Validation features shape:", val_features.shape)
print("Training targets shape:", train_targets.shape)
print("Validation targets shape:", val_targets.shape)

print("NaN in train_features:", np.isnan(train_features).any())
print("Inf in train_features:", np.isinf(train_features).any())
print("NaN in train_targets:", np.isnan(train_targets).any())
print("Inf in train_targets:", np.isinf(train_targets).any())

seeds = [42, 123, 456, 789, 1011]
results = {'mse': [], 'rmse': [], 'pearson': [], 'mae': []}

Training features shape: (13884, 7060)
Validation features shape: (4614, 7060)
Training targets shape: (13884, 1)
Validation targets shape: (4614, 1)
NaN in train_features: False
Inf in train_features: False
NaN in train_targets: False
Inf in train_targets: False


In [ ]:
for seed in seeds:
    tf.random.set_seed(seed)
    np.random.seed(seed)
    
    model = Sequential()
    for i, units in enumerate(hidden_layers):
        if i == 0:
            model.add(Dense(
                units,
                input_shape=(train_features.shape[1],),
                activation=act_func,
                kernel_initializer='he_normal'))
            if input_dropout > 0:
                model.add(Dropout(float(input_dropout)))
        else:
            model.add(Dense(
                units,
                activation=act_func,
                kernel_initializer='he_normal'))
            if hidden_dropout > 0:
                model.add(Dropout(float(hidden_dropout)))
    model.add(Dense(1, activation='linear', kernel_initializer='he_normal'))

I0000 00:00:1771567444.251553 2066093 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771567444.298042 2066093 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771567444.298072 2066093 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771567444.303334 2066093 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1771567444.303373 2066093 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:02:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
model.compile(loss='mean_squared_error', optimizer=optimizer)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 1024)              7230464   
                                                                 
 dropout (Dropout)           (None, 1024)              0         
                                                                 
 dense_1 (Dense)             (None, 1024)              1049600   
                                                                 
 dropout_1 (Dropout)         (None, 1024)              0         
                                                                 
 dense_2 (Dense)             (None, 1)                 1025      
                                                                 
Total params: 8281089 (31.59 MB)
Trainable params: 8281089 (31.59 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_path = Path(f"checkpoints/final_model_seed{seed}.h5")

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=200,
        restore_best_weights=True,
        verbose=0
        ),
    ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor='val_loss',
        save_best_only=True,            
        save_weights_only=False,
        verbose=0
        )
]

In [ ]:
history = model.fit(
    train_features, train_targets,
    validation_data=(val_features, val_targets),
    epochs=epochs,
    batch_size=64,
    callbacks=callbacks,
    shuffle=True,
    verbose=0
)

Epoch 1/1000


'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring f

 21/217 [=>............................] - ETA: 0s - loss: 566.2424   

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


200/217 [==========================>...] - ETA: 0s - loss: 452.3284
Epoch 1: val_loss improved from inf to 367.69809, saving model to checkpoints/final_model.h5


/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


217/217 [==============================] - 5s 14ms/step - loss: 452.2711 - val_loss: 367.6981
Epoch 2/1000
207/217 [===========================>..] - ETA: 0s - loss: 398.1513
Epoch 2: val_loss improved from 367.69809 to 359.91412, saving model to checkpoints/final_model.h5
217/217 [==============================] - 2s 10ms/step - loss: 398.0903 - val_loss: 359.9141
Epoch 3/1000
214/217 [============================>.] - ETA: 0s - loss: 365.8736
Epoch 3: val_loss improved from 359.91412 to 355.74249, saving model to checkpoints/final_model.h5
217/217 [==============================] - 1s 6ms/step - loss: 366.7532 - val_loss: 355.7425
Epoch 4/1000
211/217 [============================>.] - ETA: 0s - loss: 345.1354
Epoch 4: val_loss improved from 355.74249 to 348.69186, saving model to checkpoints/final_model.h5
217/217 [==============================] - 1s 5ms/step - loss: 346.5169 - val_loss: 348.6919
Epoch 5/1000
209/217 [===========================>..] - ETA: 0s - loss: 325.7261
Epoch

In [ ]:
print(f"Seed {seed} - Best Val Loss: {min(history.history['val_loss']):.4f}")

Final training loss: 127.14603424072266
Final validation loss: 277.18121337890625

Best Training Loss: 125.87498474121094

Best Validation Loss: 273.6363830566406


In [ ]:
if checkpoint_path.exists():
        model = tf.keras.models.load_model(str(checkpoint_path))

test_predictions = model.predict(test_features).flatten()    
test_targets_flat = test_targets.flatten()

mse = mean_squared_error(test_targets_flat, test_predictions)
results['mse'].append(mse)
results['rmse'].append(np.sqrt(mse))
results['pearson'].append(pearsonr(test_targets_flat, test_predictions)[0])
results['mae'].append(mean_absolute_error(test_targets_flat, test_predictions))

Loading best model from checkpoint...
145/145 [==============================] - 0s 934us/step
Validation Metrics:
Mean absolute error               : 11.1042
Mean squared error                : 273.6364
Root mean squared error           : 16.5420
Pearson's correlation coefficient : 0.6439


In [ ]:
print("FINAL RESULTS (mean ± std across 5 runs) ")
for metric, values in results.items():
    print(f"{metric.upper()}: {np.mean(values):.4f} ± {np.std(values):.4f}")

143/143 [==============================] - 0s 1ms/step
Test Fold (Fold 0) Metrics:
Pearson's correlation coefficient : 0.6140
Mean absolute error               : 10.8224
